In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: Arquivo {file_path} não encontrado.")
    raise

# ==============================================================================
# 2. IDENTIFICAÇÃO DAS COLUNAS E BINARIZAÇÃO
# ==============================================================================
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]

if not cols_esp:
    print("ERRO: Nenhuma coluna de encaminhamento/especialidade encontrada.")
else:
    # Binarização Pura (0 ou 1) nas colunas originais
    for c in cols_esp:
        df[c] = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro']).astype(int)

    # Dicionário tradutor (Nome original -> Nome bonito)
    nomes_limpos = {c: c.split('=')[-1].replace(')', '').strip().capitalize() for c in cols_esp}

    # ==============================================================================
    # 3. CÁLCULOS: PESO NA FILA E RISCO POR PACIENTE
    # ==============================================================================
    somas_absolutas = df[cols_esp].sum()
    total_encaminhamentos_gerados = somas_absolutas.sum()

    # Descobre o Top 5 Especialidades
    top_5_original = somas_absolutas.sort_values(ascending=False).head(5)

    # VISÃO A (Peso na Fila: Proporção sobre o total de guias)
    peso_na_fila = (top_5_original / total_encaminhamentos_gerados) * 100
    peso_na_fila.index = [nomes_limpos[c] for c in top_5_original.index]

    # VISÃO B (Risco por Paciente: Proporção de indivíduos únicos que precisaram)
    df_pacientes = df.groupby('Record ID')[list(top_5_original.index)].max()
    total_pacientes_unicos = df['Record ID'].nunique()
    pacientes_no_top5 = df_pacientes.sum()

    risco_por_paciente = (pacientes_no_top5 / total_pacientes_unicos) * 100
    risco_por_paciente.index = [nomes_limpos[c] for c in top_5_original.index]

    # ==============================================================================
    # 4. EXPORTAÇÃO DAS BASES DE DADOS (CSVs PARA O ARTIGO)
    # ==============================================================================
    
    # Base 1: Tabela Consolidada de Demanda (Cruza Visão A e Visão B)
    tabela_demanda = pd.DataFrame({
        'Especialidade': peso_na_fila.index,
        'Guias Emitidas (N)': top_5_original.values,
        'Peso na Fila Total (%)': peso_na_fila.values.round(2),
        'Pacientes Únicos Atendidos (N)': pacientes_no_top5.values,
        'Risco na População (%)': risco_por_paciente.values.round(2)
    })
    tabela_demanda.to_csv('base_demanda_especialidades.csv', index=False, encoding='utf-8-sig')

    # Base 2: Tabela de Auditoria Clínica (CIAP-2)
    dados_auditoria_ciap = []
    if cols_ciap:
        col_ciap_alvo = cols_ciap[0]
        # Exporta o Top 5 motivos CIAP para as Top 5 Especialidades
        for col_original in top_5_original.index:
            esp_nome = nomes_limpos[col_original]
            df_filtrado = df[df[col_original] == 1]
            total_casos = len(df_filtrado)

            if total_casos > 0:
                top_ciaps = df_filtrado[col_ciap_alvo].dropna().value_counts().head(5)
                for ciap, qtd in top_ciaps.items():
                    dados_auditoria_ciap.append({
                        'Especialidade Destino': esp_nome,
                        'Total de Guias da Especialidade': total_casos,
                        'Motivo CIAP-2': str(ciap).strip(),
                        'Frequência Absoluta (N)': qtd,
                        'Proporção do Gargalo (%)': round((qtd / total_casos) * 100, 2)
                    })
                    
        tabela_auditoria = pd.DataFrame(dados_auditoria_ciap)
        tabela_auditoria.to_csv('base_auditoria_ciap_especialidades.csv', index=False, encoding='utf-8-sig')

    print(f"\n{'='*80}")
    print("✅ BASES DE DADOS EXPORTADAS COM SUCESSO:")
    print("- 'base_demanda_especialidades.csv' (Dados das Visões A e B)")
    print("- 'base_auditoria_ciap_especialidades.csv' (Motivos Clínicos do Gargalo)")
    print(f"{'='*80}")

    # ==============================================================================
    # 5. MÁGICA VISUAL: CORES CONDICIONAIS PARA DESTAQUE
    # ==============================================================================
    # Função que pinta de vermelho se for Psiquiatria, e cinza se for outra coisa
    def gerar_paleta_destaque(nomes, termo_destaque='psiquiatria'):
        cor_destaque = '#e74c3c'  # Vermelho vibrante
        cor_neutra = '#bdc3c7'    # Cinza claro/neutro
        return [cor_destaque if termo_destaque in str(nome).lower() else cor_neutra for nome in nomes]

    paleta_a = gerar_paleta_destaque(peso_na_fila.index)
    paleta_b = gerar_paleta_destaque(risco_por_paciente.index)

    # ==============================================================================
    # 6. VISUALIZAÇÃO: PAINEL DUPLO DESTACADO
    # ==============================================================================
    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # --- GRÁFICO A: O Peso na Fila ---
    sns.barplot(x=peso_na_fila.values, y=peso_na_fila.index, palette=paleta_a, ax=ax1)
    ax1.set_title(f"VISÃO A: O Peso na Fila\n(Total: {int(total_encaminhamentos_gerados)} encaminhamentos gerados)", fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel("% de Participação no Total de Fila", fontsize=12, fontweight='bold')
    ax1.set_ylabel("")

    for i, p in enumerate(ax1.patches):
        cor_texto = '#c0392b' if paleta_a[i] == '#e74c3c' else '#333333'
        ax1.annotate(f"{p.get_width():.1f}%", (p.get_width(), p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontweight='bold', color=cor_texto)

    # --- GRÁFICO B: O Risco por Paciente ---
    sns.barplot(x=risco_por_paciente.values, y=risco_por_paciente.index, palette=paleta_b, ax=ax2)
    ax2.set_title(f"VISÃO B: A Demanda por Paciente\n(Base: {total_pacientes_unicos} pacientes únicos)", fontsize=14, fontweight='bold', pad=15)
    ax2.set_xlabel("% de Pacientes Únicos que Precisaram", fontsize=12, fontweight='bold')
    ax2.set_ylabel("")

    for i, p in enumerate(ax2.patches):
        cor_texto = '#c0392b' if paleta_b[i] == '#e74c3c' else '#333333'
        ax2.annotate(f"{p.get_width():.1f}%", (p.get_width(), p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontweight='bold', color=cor_texto)

    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 7. AUDITORIA CLÍNICA (CONSOLE)
    # ==============================================================================
    print(f"\n{'='*90}")
    print("AUDITORIA CLÍNICA: MOTIVOS DO GARGALO (TOP 3 ESPECIALIDADES)")
    print(f"{'='*90}")

    if not cols_ciap:
        print("Coluna CIAP não encontrada.")
    else:
        col_ciap_alvo = cols_ciap[0]

        # Mantemos o Top 3 no console para leitura rápida
        for col_original in top_5_original.head(3).index:
            esp_nome_bonito = nomes_limpos[col_original]
            df_filtrado = df[df[col_original] == 1]
            total_casos = len(df_filtrado)

            # Marca com asterisco se for a Psiquiatria
            marcador = " ⭐ [DESTAQUE] " if "psiquiatria" in esp_nome_bonito.lower() else " "

            print(f"\n{marcador}[ FILA: {esp_nome_bonito.upper()} ] - {total_casos} Guias Emitidas")

            if total_casos > 0:
                top_ciaps = df_filtrado[col_ciap_alvo].dropna().value_counts().head(4)
                if len(top_ciaps) > 0:
                    for ciap, qtd in top_ciaps.items():
                        perc = (qtd / total_casos) * 100
                        print(f"  -> {str(ciap)[:65]:<65} | {qtd} casos ({perc:.1f}% das guias)")
                else:
                    print("  -> Sem CIAP-2 registrado para estas guias.")
    print(f"\n{'='*90}\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: Arquivo {file_path} não encontrado.")
    raise

# ==============================================================================
# 2. RESGATE DEMOGRÁFICO BLINDADO (Idade e Sexo)
# ==============================================================================
cols_idade = [c for c in df.columns if 'idade' in c.lower()]
col_idade = cols_idade[0] if cols_idade else None

cols_sexo = [c for c in df.columns if 'sexo' in c.lower() or 'gênero' in c.lower() or 'genero' in c.lower()]
col_sexo = cols_sexo[0] if cols_sexo else None

# Espalha a idade e o sexo para a frente e para trás, preenchendo todos os retornos do paciente
if col_idade:
    df[col_idade] = df.groupby('Record ID')[col_idade].transform(lambda x: x.ffill().bfill())
if col_sexo:
    df[col_sexo] = df.groupby('Record ID')[col_sexo].transform(lambda x: x.ffill().bfill())

# Garante que o CIAP também viaje pelas linhas do paciente no mesmo dia
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
col_ciap_alvo = cols_ciap[0] if cols_ciap else None
if col_ciap_alvo:
    df[col_ciap_alvo] = df.groupby('Record ID')[col_ciap_alvo].transform(lambda x: x.ffill())

# ==============================================================================
# 3. IDENTIFICAÇÃO INTEGRAL DA SAÚDE MENTAL
# ==============================================================================
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]

# Apanha TODAS as colunas relacionadas, e não apenas a primeira
cols_psiq = [c for c in cols_esp if 'psiquiatria' in c.lower() or 'saúde mental' in c.lower() or 'psicologia' in c.lower()]

if not cols_psiq:
    print("ERRO: Nenhuma coluna de Psiquiatria/Psicologia encontrada.")
else:
    # Binariza todas as opções encontradas
    for c in cols_psiq:
        df[c] = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro']).astype(int)

    # Se marcou qualquer uma das caixas, recebe 1
    df['Fila_SaudeMental'] = df[cols_psiq].max(axis=1)

    # ==============================================================================
    # 4. PREPARAÇÃO DO DOSSIÊ POR PACIENTE ÚNICO
    # ==============================================================================
    # Filtra as linhas onde o encaminhamento aconteceu
    df_guias = df[df['Fila_SaudeMental'] == 1].copy()

    # Mantém a última guia emitida para cada paciente (para analisar indivíduos únicos)
    df_psiq = df_guias.sort_values('Repeat Instance').groupby('Record ID').last().reset_index()

    total_pacientes_psiq = len(df_psiq)

    print(f"\n{'='*90}")
    print(f"PROCESSANDO DOSSIÊ: {total_pacientes_psiq} pacientes únicos encontrados na fila.")
    print(f"Colunas unificadas: {[c.split('=')[-1].replace(')', '').strip() for c in cols_psiq]}")
    print(f"{'='*90}")

    if total_pacientes_psiq > 0:
        # Prepara a idade numérica para o gráfico
        if col_idade:
            df_psiq['Idade_Num'] = pd.to_numeric(df_psiq[col_idade], errors='coerce')

        # Carga de Comorbidades
        cols_inf = [c for c in df.columns if 'doenças infecciosas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]
        cols_cro = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]

        def to_bin(val): return 1 if str(val).lower().strip() in ['checked', '1', '1.0', 'sim', 'verdadeiro'] else 0
        for c in cols_inf + cols_cro: df_psiq[c] = df_psiq[c].apply(to_bin)

        df_psiq['Tot_Infecciosas'] = df_psiq[cols_inf].sum(axis=1)
        df_psiq['Tot_Cronicas'] = df_psiq[cols_cro].sum(axis=1)

        # Mapa Químico
        text_cols = [c for c in df_psiq.columns if df_psiq[c].dtype == object]
        df_psiq['Uso_Maconha'] = df_psiq[text_cols].apply(lambda x: x.astype(str).str.contains(r'\b(maconha|cannabis)\b', case=False, na=False)).any(axis=1).astype(int)
        df_psiq['Uso_Pesadas'] = df_psiq[text_cols].apply(lambda x: x.astype(str).str.contains(r'\b(crack|coca[íi]na|hero[íi]na|opi[óo]ide)\b', case=False, na=False)).any(axis=1).astype(int)

        cols_alcool = [c for c in df.columns if 'álcool' in c.lower() or 'alcool' in c.lower() and 'choice=' in c.lower()]
        cols_tabaco = [c for c in df.columns if ('tabaco' in c.lower() or 'fumo' in c.lower() or 'cigarro' in c.lower()) and 'choice=' in c.lower()]

        df_psiq['Uso_Alcool'] = df_psiq[cols_alcool].apply(lambda col: col.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_alcool else 0
        df_psiq['Uso_Tabaco'] = df_psiq[cols_tabaco].apply(lambda col: col.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_tabaco else 0

        # ==============================================================================
        # 4.5. EXPORTAÇÃO DAS BASES DE DADOS (CSVs PARA O ARTIGO)
        # ==============================================================================
        
        # Base 1: Motivos CIAP-2
        if col_ciap_alvo:
            tabela_ciap = df_psiq[col_ciap_alvo].dropna().value_counts().reset_index()
            tabela_ciap.columns = ['Motivo CIAP-2', 'Pacientes Únicos (N)']
            tabela_ciap['Proporção (%)'] = (tabela_ciap['Pacientes Únicos (N)'] / total_pacientes_psiq * 100).round(2)
            tabela_ciap.to_csv('base_dossie_motivos_psiq.csv', index=False, encoding='utf-8-sig')

        # Base 2: Demografia (Estatísticas de Idade por Sexo)
        if col_idade and col_sexo:
            df_demog = df_psiq.dropna(subset=['Idade_Num', col_sexo])
            if len(df_demog) > 0:
                top_sexos = df_demog[col_sexo].value_counts().head(2).index
                df_demog_filtrado = df_demog[df_demog[col_sexo].isin(top_sexos)]
                tabela_demog = df_demog_filtrado.groupby(col_sexo)['Idade_Num'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).reset_index()
                tabela_demog.columns = ['Sexo', 'N (Pacientes)', 'Idade Média', 'Idade Mediana', 'Desvio Padrão', 'Idade Mínima', 'Idade Máxima']
                tabela_demog['Idade Média'] = tabela_demog['Idade Média'].round(1)
                tabela_demog['Desvio Padrão'] = tabela_demog['Desvio Padrão'].round(2)
                tabela_demog.to_csv('base_dossie_demografia_psiq.csv', index=False, encoding='utf-8-sig')

        # Base 3: Carga de Comorbidades Físicas
        comorbidades = pd.DataFrame({
            'Categoria de Carga': ['Nenhuma Doença', '1 a 2 Doenças', '3+ Doenças'],
            'Infecciosas (N)': [
                len(df_psiq[df_psiq['Tot_Infecciosas'] == 0]),
                len(df_psiq[(df_psiq['Tot_Infecciosas'] >= 1) & (df_psiq['Tot_Infecciosas'] <= 2)]),
                len(df_psiq[df_psiq['Tot_Infecciosas'] >= 3])
            ],
            'Crônicas (N)': [
                len(df_psiq[df_psiq['Tot_Cronicas'] == 0]),
                len(df_psiq[(df_psiq['Tot_Cronicas'] >= 1) & (df_psiq['Tot_Cronicas'] <= 2)]),
                len(df_psiq[df_psiq['Tot_Cronicas'] >= 3])
            ]
        })
        comorbidades.to_csv('base_dossie_comorbidades_psiq.csv', index=False, encoding='utf-8-sig')

        # Base 4: Mapa Químico (Uso de Substâncias)
        uso_substancias = {
            'Álcool': df_psiq['Uso_Alcool'].sum(),
            'Tabaco': df_psiq['Uso_Tabaco'].sum(),
            'Maconha': df_psiq['Uso_Maconha'].sum(),
            'Drogas Pesadas (Crack/Cocaína/Opioides)': df_psiq['Uso_Pesadas'].sum()
        }
        tabela_substancias = pd.DataFrame(list(uso_substancias.items()), columns=['Substância', 'Pacientes Únicos (N)'])
        tabela_substancias['Proporção (%)'] = (tabela_substancias['Pacientes Únicos (N)'] / total_pacientes_psiq * 100).round(2)
        tabela_substancias = tabela_substancias.sort_values('Pacientes Únicos (N)', ascending=False)
        tabela_substancias.to_csv('base_dossie_substancias_psiq.csv', index=False, encoding='utf-8-sig')

        print(f"\n{'='*80}")
        print("✅ BASES DE DADOS DO DOSSIÊ EXPORTADAS COM SUCESSO:")
        if col_ciap_alvo: print("- 'base_dossie_motivos_psiq.csv'")
        if col_idade and col_sexo: print("- 'base_dossie_demografia_psiq.csv'")
        print("- 'base_dossie_comorbidades_psiq.csv'")
        print("- 'base_dossie_substancias_psiq.csv'")
        print(f"{'='*80}")

        # ==============================================================================
        # 5. O DASHBOARD (4 PAINÉIS)
        # ==============================================================================
        sns.set_theme(style="whitegrid")
        fig = plt.figure(figsize=(20, 14))
        fig.suptitle(f"Dossiê Epidemiológico: Fila de Saúde Mental (N = {total_pacientes_psiq} pacientes únicos)", fontsize=20, fontweight='bold', y=0.98)

        # PAINEL 1: MOTIVOS
        ax1 = plt.subplot(2, 2, 1)
        if col_ciap_alvo:
            top_ciaps = df_psiq[col_ciap_alvo].dropna().value_counts().head(5)
            nomes_curtos = [str(m)[:50] + ('...' if len(str(m)) > 50 else '') for m in top_ciaps.index]
            sns.barplot(x=top_ciaps.values, y=nomes_curtos, palette="Reds_r", ax=ax1)
            ax1.set_title("1. Principais Motivos de Encaminhamento (CIAP-2)", fontsize=14, fontweight='bold')
            ax1.set_xlabel("Número de Pacientes Únicos")
            for p in ax1.patches:
                ax1.annotate(f" {int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2.), va='center', fontweight='bold')

        # PAINEL 2: DEMOGRAFIA (IDADE PREENCHIDA)
        ax2 = plt.subplot(2, 2, 2)
        if col_idade and col_sexo:
            df_demog = df_psiq.dropna(subset=['Idade_Num', col_sexo])
            if len(df_demog) > 0:
                top_sexos = df_demog[col_sexo].value_counts().head(2).index
                df_demog_filtrado = df_demog[df_demog[col_sexo].isin(top_sexos)]
                sns.violinplot(data=df_demog_filtrado, x=col_sexo, y='Idade_Num', palette="muted", inner="quart", ax=ax2)
                ax2.set_title("2. Perfil Demográfico: Distribuição de Idade por Sexo", fontsize=14, fontweight='bold')
                ax2.set_ylabel("Idade (Anos)")
                ax2.set_xlabel("")
            else:
                ax2.text(0.5, 0.5, "Idades numéricas não encontradas para os pacientes", ha='center', va='center')

        # PAINEL 3: COMORBIDADES
        ax3 = plt.subplot(2, 2, 3)
        comorb_melt = comorbidades.melt(id_vars='Categoria de Carga', var_name='Tipo', value_name='Pacientes')
        sns.barplot(data=comorb_melt, x='Categoria de Carga', y='Pacientes', hue='Tipo', palette=['#e67e22', '#2980b9'], ax=ax3)
        ax3.set_title("3. Carga de Doenças Físicas (Comorbidades)", fontsize=14, fontweight='bold')
        ax3.set_ylabel("Número de Pacientes")

        # PAINEL 4: MAPA QUÍMICO
        ax4 = plt.subplot(2, 2, 4)
        sr_substancias = pd.Series(uso_substancias).sort_values(ascending=False)
        sns.barplot(x=sr_substancias.values, y=sr_substancias.index, palette="viridis", ax=ax4)
        ax4.set_title("4. Mapa Químico: Uso de Substâncias na Fila", fontsize=14, fontweight='bold')
        ax4.set_xlabel("Número de Pacientes Únicos")
        for p in ax4.patches:
            ax4.annotate(f" {int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2.), va='center', fontweight='bold')

        sns.despine(left=True, bottom=True)
        plt.tight_layout(pad=3.0)
        plt.show()

        # ==============================================================================
        # 6. RELATÓRIO DO CONSOLE (CIAPs E IDADES REAIS)
        # ==============================================================================
        print(f"\n[ VERIFICAÇÃO DE MOTIVOS (CIAP-2) ]")
        if col_ciap_alvo:
            top_ciaps_console = df_psiq[col_ciap_alvo].dropna().value_counts().head(8)
            for ciap, qtd in top_ciaps_console.items():
                perc = (qtd / total_pacientes_psiq) * 100
                print(f"  -> {str(ciap)[:70]:<70} | {qtd} pacientes ({perc:.1f}%)")

        print(f"\n[ VERIFICAÇÃO DE IDADES ]")
        if col_idade and len(df_demog) > 0:
            print(f"  -> Idade Média calculada: {df_demog['Idade_Num'].mean():.1f} anos")
            print(f"  -> Paciente mais novo: {df_demog['Idade_Num'].min():.0f} anos")
            print(f"  -> Paciente mais velho: {df_demog['Idade_Num'].max():.0f} anos")
        print(f"{'='*90}\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# ==============================================================================
# 2. RESGATE DEMOGRÁFICO BLINDADO (Idade e Sexo)
# ==============================================================================
cols_idade = [c for c in df.columns if 'idade' in c.lower()]
col_idade = cols_idade[0] if cols_idade else None

cols_sexo = [c for c in df.columns if 'sexo' in c.lower() or 'gênero' in c.lower() or 'genero' in c.lower()]
col_sexo = cols_sexo[0] if cols_sexo else None

# Espalha a idade e o sexo para todos os retornos do paciente
if col_idade:
    df[col_idade] = df.groupby('Record ID')[col_idade].transform(lambda x: x.ffill().bfill())
if col_sexo:
    df[col_sexo] = df.groupby('Record ID')[col_sexo].transform(lambda x: x.ffill().bfill())

# Espalha o CIAP
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
col_ciap_alvo = cols_ciap[0] if cols_ciap else None
if col_ciap_alvo:
    df[col_ciap_alvo] = df.groupby('Record ID')[col_ciap_alvo].transform(lambda x: x.ffill())

# ==============================================================================
# 3. IDENTIFICAÇÃO DAS ESPECIALIDADES ALVO
# ==============================================================================
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]

especialidades_alvo = ['oftalmologia', 'ortopedia']

for alvo in especialidades_alvo:
    # Busca a coluna exata da especialidade atual no loop
    col_alvo = next((c for c in cols_esp if alvo in c.lower()), None)

    if not col_alvo:
        print(f"\n[ERRO] Coluna para a especialidade '{alvo.upper()}' não encontrada na base.")
        continue

    # Binariza (0 ou 1)
    df['Fila_Atual'] = df[col_alvo].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro']).astype(int)

    # ==============================================================================
    # 4. PREPARAÇÃO DO DOSSIÊ POR PACIENTE ÚNICO (Para a fila atual)
    # ==============================================================================
    df_guias = df[df['Fila_Atual'] == 1].copy()

    # Mantém a última guia emitida para cada paciente para analisar indivíduos únicos
    df_alvo = df_guias.sort_values('Repeat Instance').groupby('Record ID').last().reset_index()

    total_pacientes = len(df_alvo)

    print(f"\n{'='*100}")
    print(f"GERANDO DOSSIÊ E BASES: {alvo.upper()} | {total_pacientes} pacientes únicos na fila.")
    print(f"{'='*100}")

    if total_pacientes > 0:
        # Prepara Idade Numérica
        if col_idade:
            df_alvo['Idade_Num'] = pd.to_numeric(df_alvo[col_idade], errors='coerce')

        # Carga de Comorbidades Físicas
        cols_inf = [c for c in df.columns if 'doenças infecciosas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]
        cols_cro = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]

        def to_bin(val): return 1 if str(val).lower().strip() in ['checked', '1', '1.0', 'sim', 'verdadeiro'] else 0
        for c in cols_inf + cols_cro: df_alvo[c] = df_alvo[c].apply(to_bin)

        df_alvo['Tot_Infecciosas'] = df_alvo[cols_inf].sum(axis=1)
        df_alvo['Tot_Cronicas'] = df_alvo[cols_cro].sum(axis=1)

        # Mapa Químico
        text_cols = [c for c in df_alvo.columns if df_alvo[c].dtype == object]
        df_alvo['Uso_Maconha'] = df_alvo[text_cols].apply(lambda x: x.astype(str).str.contains(r'\b(maconha|cannabis)\b', case=False, na=False)).any(axis=1).astype(int)
        df_alvo['Uso_Pesadas'] = df_alvo[text_cols].apply(lambda x: x.astype(str).str.contains(r'\b(crack|coca[íi]na|hero[íi]na|opi[óo]ide)\b', case=False, na=False)).any(axis=1).astype(int)

        cols_alcool = [c for c in df.columns if 'álcool' in c.lower() or 'alcool' in c.lower() and 'choice=' in c.lower()]
        cols_tabaco = [c for c in df.columns if ('tabaco' in c.lower() or 'fumo' in c.lower() or 'cigarro' in c.lower()) and 'choice=' in c.lower()]

        df_alvo['Uso_Alcool'] = df_alvo[cols_alcool].apply(lambda col: col.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_alcool else 0
        df_alvo['Uso_Tabaco'] = df_alvo[cols_tabaco].apply(lambda col: col.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_tabaco else 0

        # Prepara os DataFrames para Exportação e Plotagem
        comorbidades = pd.DataFrame({
            'Categoria de Carga': ['Nenhuma Fís.', '1 a 2 Doenças', '3+ Doenças'],
            'Infecciosas (N)': [
                len(df_alvo[df_alvo['Tot_Infecciosas'] == 0]),
                len(df_alvo[(df_alvo['Tot_Infecciosas'] >= 1) & (df_alvo['Tot_Infecciosas'] <= 2)]),
                len(df_alvo[df_alvo['Tot_Infecciosas'] >= 3])
            ],
            'Crônicas (N)': [
                len(df_alvo[df_alvo['Tot_Cronicas'] == 0]),
                len(df_alvo[(df_alvo['Tot_Cronicas'] >= 1) & (df_alvo['Tot_Cronicas'] <= 2)]),
                len(df_alvo[df_alvo['Tot_Cronicas'] >= 3])
            ]
        })

        uso_substancias = {
            'Álcool': df_alvo['Uso_Alcool'].sum(),
            'Tabaco': df_alvo['Uso_Tabaco'].sum(),
            'Maconha': df_alvo['Uso_Maconha'].sum(),
            'Drogas Pesadas (Crack/Cocaína/Opioides)': df_alvo['Uso_Pesadas'].sum()
        }

        # ==============================================================================
        # 4.5 EXPORTAÇÃO DAS BASES DE DADOS (CSVs) PARA A ESPECIALIDADE
        # ==============================================================================
        
        # Base 1: Motivos CIAP-2
        if col_ciap_alvo:
            tabela_ciap = df_alvo[col_ciap_alvo].dropna().value_counts().reset_index()
            tabela_ciap.columns = ['Motivo CIAP-2', 'Pacientes Únicos (N)']
            tabela_ciap['Proporção (%)'] = (tabela_ciap['Pacientes Únicos (N)'] / total_pacientes * 100).round(2)
            tabela_ciap.to_csv(f'base_dossie_motivos_{alvo}.csv', index=False, encoding='utf-8-sig')

        # Base 2: Demografia (Idade por Sexo)
        if col_idade and col_sexo:
            df_demog_export = df_alvo.dropna(subset=['Idade_Num', col_sexo])
            if len(df_demog_export) > 0:
                top_sexos_export = df_demog_export[col_sexo].value_counts().head(2).index
                df_demog_filtrado = df_demog_export[df_demog_export[col_sexo].isin(top_sexos_export)]
                tabela_demog = df_demog_filtrado.groupby(col_sexo)['Idade_Num'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).reset_index()
                tabela_demog.columns = ['Sexo', 'N (Pacientes)', 'Idade Média', 'Idade Mediana', 'Desvio Padrão', 'Idade Mínima', 'Idade Máxima']
                tabela_demog['Idade Média'] = tabela_demog['Idade Média'].round(1)
                tabela_demog['Desvio Padrão'] = tabela_demog['Desvio Padrão'].round(2)
                tabela_demog.to_csv(f'base_dossie_demografia_{alvo}.csv', index=False, encoding='utf-8-sig')

        # Base 3: Comorbidades
        comorbidades.to_csv(f'base_dossie_comorbidades_{alvo}.csv', index=False, encoding='utf-8-sig')

        # Base 4: Mapa Químico
        tabela_substancias = pd.DataFrame(list(uso_substancias.items()), columns=['Substância', 'Pacientes Únicos (N)'])
        tabela_substancias['Proporção (%)'] = (tabela_substancias['Pacientes Únicos (N)'] / total_pacientes * 100).round(2)
        tabela_substancias = tabela_substancias.sort_values('Pacientes Únicos (N)', ascending=False)
        tabela_substancias.to_csv(f'base_dossie_substancias_{alvo}.csv', index=False, encoding='utf-8-sig')

        print(f"✅ BASES DE DADOS EXPORTADAS ({alvo.upper()}):")
        print(f" - 'base_dossie_motivos_{alvo}.csv'")
        print(f" - 'base_dossie_demografia_{alvo}.csv'")
        print(f" - 'base_dossie_comorbidades_{alvo}.csv'")
        print(f" - 'base_dossie_substancias_{alvo}.csv'")

        # ==============================================================================
        # 5. O DASHBOARD (4 PAINÉIS)
        # ==============================================================================
        sns.set_theme(style="whitegrid")
        fig = plt.figure(figsize=(20, 14))

        # Cor de destaque baseada na especialidade
        cor_destaque = '#2980b9' if alvo == 'oftalmologia' else '#27ae60'

        fig.suptitle(f"Dossiê Epidemiológico: Fila de {alvo.upper()} (N = {total_pacientes} pacientes únicos)", fontsize=20, fontweight='bold', y=0.98, color=cor_destaque)

        # PAINEL 1: MOTIVOS (CIAP-2)
        ax1 = plt.subplot(2, 2, 1)
        if col_ciap_alvo:
            top_ciaps = df_alvo[col_ciap_alvo].dropna().value_counts().head(5)
            nomes_curtos = [str(m)[:50] + ('...' if len(str(m)) > 50 else '') for m in top_ciaps.index]
            sns.barplot(x=top_ciaps.values, y=nomes_curtos, palette=f"light:{cor_destaque}_r", ax=ax1)
            ax1.set_title(f"1. Principais Motivos (CIAP-2) para {alvo.capitalize()}", fontsize=14, fontweight='bold')
            ax1.set_xlabel("Número de Pacientes Únicos")
            for p in ax1.patches:
                ax1.annotate(f" {int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2.), va='center', fontweight='bold')

        # PAINEL 2: DEMOGRAFIA (IDADE PREENCHIDA)
        ax2 = plt.subplot(2, 2, 2)
        if col_idade and col_sexo:
            df_demog = df_alvo.dropna(subset=['Idade_Num', col_sexo])
            if len(df_demog) > 0:
                top_sexos = df_demog[col_sexo].value_counts().head(2).index
                df_demog = df_demog[df_demog[col_sexo].isin(top_sexos)]
                sns.violinplot(data=df_demog, x=col_sexo, y='Idade_Num', palette="muted", inner="quart", ax=ax2)
                ax2.set_title("2. Perfil Demográfico: Idade vs. Sexo", fontsize=14, fontweight='bold')
                ax2.set_ylabel("Idade (Anos)")
                ax2.set_xlabel("")

        # PAINEL 3: COMORBIDADES
        ax3 = plt.subplot(2, 2, 3)
        comorb_melt = comorbidades.melt(id_vars='Categoria de Carga', var_name='Tipo', value_name='Pacientes')
        sns.barplot(data=comorb_melt, x='Categoria de Carga', y='Pacientes', hue='Tipo', palette=['#e67e22', cor_destaque], ax=ax3)
        ax3.set_title("3. Carga Concorrente de Doenças (Infecciosas vs Crônicas)", fontsize=14, fontweight='bold')
        ax3.set_ylabel("Número de Pacientes")

        # PAINEL 4: MAPA QUÍMICO
        ax4 = plt.subplot(2, 2, 4)
        sr_substancias = pd.Series(uso_substancias).sort_values(ascending=False)
        sns.barplot(x=sr_substancias.values, y=sr_substancias.index, palette="viridis", ax=ax4)
        ax4.set_title("4. Mapa Químico na Fila", fontsize=14, fontweight='bold')
        ax4.set_xlabel("Número de Pacientes Únicos")
        for p in ax4.patches:
            ax4.annotate(f" {int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2.), va='center', fontweight='bold')

        sns.despine(left=True, bottom=True)
        plt.tight_layout(pad=3.0)
        plt.show()

        # ==============================================================================
        # 6. RELATÓRIO DO CONSOLE (CIAPs E IDADES REAIS)
        # ==============================================================================
        print(f"\n[ VERIFICAÇÃO DE MOTIVOS (CIAP-2) - {alvo.upper()} ]")
        if col_ciap_alvo:
            top_ciaps_console = df_alvo[col_ciap_alvo].dropna().value_counts().head(5)
            for ciap, qtd in top_ciaps_console.items():
                perc = (qtd / total_pacientes) * 100
                print(f"  -> {str(ciap)[:70]:<70} | {qtd} pacientes ({perc:.1f}%)")

        print(f"\n[ VERIFICAÇÃO DE IDADES - {alvo.upper()} ]")
        if col_idade and len(df_demog) > 0:
            print(f"  -> Idade Média calculada: {df_demog['Idade_Num'].mean():.1f} anos")
            print(f"  -> Paciente mais novo: {df_demog['Idade_Num'].min():.0f} anos")
            print(f"  -> Paciente mais velho: {df_demog['Idade_Num'].max():.0f} anos")
        print(f"{'='*100}\n")